<center><img src="https://keras.io/img/logo-small.png" alt="Keras logo" width="100"><br/>
This starter notebook is provided by the Keras team.</center>

# Automated Essay Scoring 2.0 with [KerasNLP](https://github.com/keras-team/keras-nlp) and [Keras](https://github.com/keras-team/keras)

<div align="center">
    <img src="https://i.ibb.co/BrZf1MC/AESv2.jpg">
</div>

In this competition, our aim is to develop an AI model that can score student essays. This competition is actually an updated version of an old one that took place over a decade ago. In this version, we aim to improve upon essay scoring algorithms to enhance student learning outcomes. This notebook will guide you through the process of fine-tuning the **DebertaV3** model using **Ordinal Regression/Classification** to score student essays using KerasNLP.

**Did you know**: This notebook is backend-agnostic, which means it supports TensorFlow, PyTorch, and JAX backends. However, the best performance can be achieved with `JAX`. KerasNLP and Keras enable the choice of the preferred backend. Explore further details on [Keras](https://keras.io/keras_3/).


**Note**: For a deeper understanding of KerasNLP, refer to the [KerasNLP guides](https://keras.io/keras_nlp/).

# 📚 | Import Libraries 

In [ ]:
import os
print(os.listdir('/kaggle/input'))
print(os.listdir('/kaggle/input/datasets'))
print(os.listdir('/kaggle/input/datasets/nbroad'))
print(os.listdir('/kaggle/input/datasets/nbroad/persaude-corpus-2'))

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"  # "jax" or "tensorflow" or "torch" 

import keras_nlp
import keras
import keras.backend as K
import tensorflow as tf

import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

cmap = mpl.cm.get_cmap('coolwarm')

## Library Version

In [ ]:
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("KerasNLP:", keras_nlp.__version__)

# ⚙️ | Configuration

## Screenshot 1 - Experiment Parameters

**Purpose:** Show the main hyperparameters used in the project, including LGBM settings, Transformer sequence length, batch size, dropout, and training epochs.

In [ ]:
class CFG:
    seed = 42  # Random seed
    use_lgbm = True  # True: train LGBM; False: train Transformer

    # Transformer settings
    #preset = "deberta_v3_extra_small_en" # Name of pretrained models
    preset = "roberta_base_en"
    #preset = "bert_base_en_uncased"
    sequence_length = 512  # Input sequence length
    epochs = 3 # Training epochs
    batch_size = 32  # Batch size
    scheduler = 'cosine'  # Learning rate scheduler
    head_units = 64
    head_dropout = 0.25
    head_samples = 5

    # LGBM settings
    lgbm_word_max_features = 30000
    lgbm_char_max_features = 50000
    lgbm_learning_rate = 0.03
    lgbm_estimators = 2500
    lgbm_early_stopping_rounds = 100
    lgbm_optimize_thresholds = True


## Model Run Switch Board

Use this single cell to choose which model(s) to run on Kaggle. Original baseline models run one at a time; added models can run as a comparison set.

In [ ]:
# ===== Total Model Switch Board: edit only this cell before running on Kaggle =====
# Original / teammate baseline models: choose at most ONE True.
# Added models from Huang Po-Shuo's experiment can be combined.

RUN_MODELS = {
    # Original baseline section
    "lgbm": True,
    "bert": False,
    "roberta": False,
    "deberta_v3_small": False,
    "deberta_v3_base": False,

    # Added model comparison section
    "longformer": True,
    "electra": True,
    "albert": False,
}

# Choose which added model should create submission.csv when ensemble is disabled.
# When optimized ensemble is enabled below, ensemble submission has first priority.
SUBMISSION_MODEL = "longformer"

ORIGINAL_MODEL_CONFIGS = {
    "lgbm": {
        "preset": None,
        "model_family": "lgbm",
        "preprocessor_cls": None,
        "classifier_cls": None,
    },
    "bert": {
        "preset": "bert_base_en_uncased",
        "model_family": "bert",
        "preprocessor_cls": "BertPreprocessor",
        "classifier_cls": "BertClassifier",
    },
    "roberta": {
        "preset": "roberta_base_en",
        "model_family": "roberta",
        "preprocessor_cls": "RobertaPreprocessor",
        "classifier_cls": "RobertaClassifier",
    },
    "deberta_v3_small": {
        "preset": "deberta_v3_extra_small_en",
        "model_family": "deberta_v3",
        "preprocessor_cls": "DebertaV3Preprocessor",
        "classifier_cls": "DebertaV3Classifier",
    },
    "deberta_v3_base": {
        "preset": "deberta_v3_base_en",
        "model_family": "deberta_v3",
        "preprocessor_cls": "DebertaV3Preprocessor",
        "classifier_cls": "DebertaV3Classifier",
    },
}

original_order = ["lgbm", "bert", "roberta", "deberta_v3_small", "deberta_v3_base"]
added_order = ["longformer", "electra", "albert"]
selected_original = [name for name in original_order if RUN_MODELS.get(name, False)]
HF_ENABLED_MODELS = [name for name in added_order if RUN_MODELS.get(name, False)]

if len(selected_original) > 1:
    print("More than one original baseline is enabled; using the first one only:", selected_original[0])
selected_original = selected_original[:1]

CFG.run_original_model = len(selected_original) == 1
CFG.active_original_model = selected_original[0] if selected_original else None

if CFG.run_original_model:
    original_cfg = ORIGINAL_MODEL_CONFIGS[CFG.active_original_model]
    CFG.use_lgbm = CFG.active_original_model == "lgbm"
    CFG.preset = original_cfg["preset"]
    CFG.model_family = original_cfg["model_family"]
    CFG.preprocessor_cls = original_cfg["preprocessor_cls"]
    CFG.classifier_cls = original_cfg["classifier_cls"]
else:
    CFG.use_lgbm = False
    CFG.preset = None
    CFG.model_family = None
    CFG.preprocessor_cls = None
    CFG.classifier_cls = None

HF_SUBMISSION_MODEL = SUBMISSION_MODEL if SUBMISSION_MODEL in HF_ENABLED_MODELS else (HF_ENABLED_MODELS[0] if HF_ENABLED_MODELS else None)

print("Original baseline enabled:", CFG.active_original_model if CFG.run_original_model else "None")
print("Added models enabled:", HF_ENABLED_MODELS if HF_ENABLED_MODELS else "None")
print("Submission model:", HF_SUBMISSION_MODEL if HF_SUBMISSION_MODEL else (CFG.active_original_model if CFG.run_original_model else "None"))


# ♻️ | Reproducibility 
Sets value for random seed to produce similar result in each run.

In [ ]:
keras.utils.set_random_seed(CFG.seed)

# 🧮 | Mixed Precision

In [ ]:
keras.mixed_precision.set_global_policy("mixed_float16")

# 📁 | Dataset Path 

In [ ]:
# ===== Kaggle Data Path Auto-Detect =====
# This avoids FileNotFoundError when Kaggle changes the competition input folder name.
import os

BASE_PATH_CANDIDATES = [
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2",
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2-0",
]

BASE_PATH = None
for candidate in BASE_PATH_CANDIDATES:
    if os.path.exists(os.path.join(candidate, "train.csv")):
        BASE_PATH = candidate
        break

if BASE_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(set(files)):
            BASE_PATH = root
            break

if BASE_PATH is None:
    raise FileNotFoundError(
        "Could not find train.csv/test.csv/sample_submission.csv under /kaggle/input. "
        "Please add the competition data with Add Input -> Competitions."
    )

print("Using competition data path:", BASE_PATH)


# 📖 | Meta Data

**Files in the dataset:**

- `{test|train}.csv`
  - `essay_id`: Unique identifier for each essay.
  - `full_text`: Essay text.
  - `score`: Essay's score from `1-6`.
- `sample_submission.csv`: Valid sample submission.

**What does the `score` mean?**

The `score` represents the quality of student-written argumentative essays. Essays were rated based on a rubric covering perspective development, critical thinking, evidence use, organization, language, and grammar/mechanics. Here's a summary of the scoring criteria:

| Score | Description |
|-------|-------------|
| 6     | Clear mastery with few errors, outstanding critical thinking, appropriate evidence, well-organized, skilled language use. |
| 5     | Reasonable mastery with occasional errors, strong critical thinking, generally appropriate evidence, well-organized, good language use. |
| 4     | Adequate mastery with some lapses, competent critical thinking, adequate evidence, generally organized, fair language use. |
| 3     | Developing mastery with weaknesses, limited critical thinking, inconsistent evidence, limited organization, fair language use with weaknesses. |
| 2     | Little mastery with serious flaws, weak critical thinking, insufficient evidence, poor organization, limited language use with frequent errors. |
| 1     | Very little or no mastery, severely flawed, no viable point of view, disorganized, fundamental language flaws, pervasive grammar/mechanics errors. |

> This grading is very similar to the grading used in the [ETS GRE (Graduate Record Examinations) AWA](https://www.ets.org/gre/test-takers/general-test/prepare/content/analytical-writing.html) exam, where prospective graduate students are asked to write essays to judge their analytical abilities, and their scores are later used for graduate admission. 


In [ ]:
# Load data
df = pd.read_csv(f'{BASE_PATH}/train.csv')  # Read CSV file into a DataFrame

# Display information about the train data
print("# Train Data: {:,}".format(len(df)))
print("# Sample:")
display(df.head(2))

In [ ]:
# 載入 PERSUADE 2.0
persuade_df = pd.read_csv('/kaggle/input/datasets/nbroad/persaude-corpus-2/persuade_2.0_human_scores_demo_id_github.csv')
# 先看欄位確認名稱正確
print(persuade_df.columns.tolist())
print(persuade_df.shape)

# 對齊欄位
persuade_df = persuade_df.rename(columns={
    'full_text': 'full_text',
    'holistic_essay_score': 'score'
})[['full_text', 'score']]

# 清理資料
persuade_df = persuade_df.dropna(subset=['full_text', 'score'])
persuade_df['score'] = persuade_df['score'].astype(int)
persuade_df = persuade_df[persuade_df['score'].between(1, 6)]

# 只保留競賽資料需要的欄位
df = df[['full_text', 'score']]

# 合併並去除重複
df = pd.concat([df, persuade_df], ignore_index=True)
df = df.drop_duplicates(subset=['full_text'])
df = df.reset_index(drop=True)

print(f"原始競賽資料：17307 筆")
print(f"PERSUADE 2.0：{len(persuade_df)} 筆")
print(f"合併後總筆數：{len(df)} 筆")

## Screenshot 2 - Text Preprocessing

**Purpose:** Show how raw essays are cleaned before modeling: lowercase conversion, HTML/link removal, whitespace normalization, contraction expansion, and head-tail truncation for Transformer models.

In [ ]:
import re

# Contraction dictionary
cList = {
    "ain't": "am not", "aren't": "are not", "can't": "cannot",
    "couldn't": "could not", "didn't": "did not", "doesn't": "does not",
    "don't": "do not", "hasn't": "has not", "haven't": "have not",
    "he's": "he is", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it's": "it is", "she's": "she is",
    "they're": "they are", "wasn't": "was not", "we're": "we are",
    "weren't": "were not", "won't": "will not", "you're": "you are",
}
c_re = re.compile('(%s)' % '|'.join(cList.keys()))

def expand_contractions(text):
    def replace(match):
        return cList[match.group(0)]
    return c_re.sub(replace, text)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    # text = re.sub(r'\d+', '', text)
    text = re.sub(r'http\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\.+', '.', text)
    text = expand_contractions(text)
    text = text.strip()
    return text

def truncate_head_tail(text, max_length=512):
    tokens = text.split()
    if len(tokens) > max_length:
        head = tokens[:256]
        tail = tokens[-256:]
        return ' '.join(head + tail)
    return text

# LGBM uses the full cleaned essay; Transformer uses the 512-token head/tail version.
df['full_text_clean'] = df['full_text'].fillna('').apply(clean_text)
df['full_text_transformer'] = df['full_text_clean'].apply(truncate_head_tail)


## Screenshot 3 - Essay Feature Engineering

**Purpose:** Show the engineered essay-level features used together with text models: word count, paragraph count, sentence count, vocabulary richness, spelling errors, and spelling error rate.

In [ ]:

#import spacy
#nlp = spacy.load("en_core_web_sm")

 
# https://www.kaggle.com/datasets/rtatman/english-word-frequency
import nltk
nltk.data.path.append('/kaggle/input/nltk-data')
from nltk.corpus import words as nltk_words
english_vocab = set(w.lower() for w in nltk_words.words())

import spacy
nlp = spacy.load("en_core_web_sm")

FEATURE_COLS = [
    "word_count",
    "paragraph_count",
    "sentence_count",
    "avg_word_per_sentence",
    "unique_word_count",
    "vocab_richness",
    "spelling_errors",
    "spelling_error_rate",
]

def count_spelling_errors(text):
    doc = nlp(text)
    errors = sum(1 for token in doc if token.lemma_.lower() not in english_vocab)
    return errors

def add_essay_features(dataframe):
    dataframe = dataframe.copy()
    text = dataframe["full_text"].fillna("")
    words = text.apply(lambda x: x.split())

    dataframe["word_count"] = words.apply(len).clip(lower=1)
    dataframe["paragraph_count"] = text.apply(lambda x: max(1, len([p for p in x.split("\n\n") if p.strip()])))
    dataframe["sentence_count"] = text.apply(lambda x: max(1, len([s for s in re.split(r"[.!?]+", x) if s.strip()])))
    dataframe["avg_word_per_sentence"] = dataframe["word_count"] / dataframe["sentence_count"]
    dataframe["unique_word_count"] = words.apply(lambda x: len(set(w.lower() for w in x)))
    dataframe["vocab_richness"] = dataframe["unique_word_count"] / dataframe["word_count"]
    dataframe["spelling_errors"] = text.apply(count_spelling_errors)
    dataframe["spelling_error_rate"] = dataframe["spelling_errors"] / dataframe["word_count"]

    dataframe[FEATURE_COLS] = dataframe[FEATURE_COLS].replace([np.inf, -np.inf], 0).fillna(0)
    return dataframe

def fit_feature_scaler(dataframe):
    mean = dataframe[FEATURE_COLS].mean()
    std = dataframe[FEATURE_COLS].std().replace(0, 1)
    return mean, std

def transform_features(dataframe, mean, std):
    features = (dataframe[FEATURE_COLS] - mean) / std
    return features.replace([np.inf, -np.inf], 0).fillna(0).astype("float32").values

# Transformer head
df = add_essay_features(df)


# 🔪 | Data Split

In the code snippet provided below, we will divide the existing **train** data into folds using a stratification of `label` column.

In [ ]:
from sklearn.model_selection import train_test_split  # Import package

train_df, valid_df = train_test_split(df, test_size=0.2, stratify=df["score"])

# 🧂 | Process Labels for Ordinal Regression

**What is Ordinal Regression?**
Ordinal regression is a type of supervised learning algorithm used to predict an ordinal or ranked dependent variable. Unlike in typical classification problems where the target variable is categorical, in ordinal regression the target variable has a ordering or ranking. It is also different from typical regression, where the target variable is continuous. In ordinal regression, the target variable has a discrete, ordered set of values, such as levels of satisfaction or performance grades.

**Why do we need it?**
Ordinal regression is useful when the target variable (in our competition, the `score` of an essay) represents a ranking or order. Specifically, ordinal regression can capture the inherent order in the target variable, which standard classification models may not be able to do effectively.

For Ordinal Regression, we have to transform the labels, which is different from typical classification or regression. The following code will convert essay scores to an ordinal matrix, which will be used as the ground truth for calculating the loss.

In [ ]:
def to_ordinal(y, num_classes=None, dtype="float32"):
    """Converts a class vector (integers) to an ordinal regression matrix.

    This utility encodes class vector to ordinal regression/classification
    matrix where each sample is indicated by a row and rank of that sample is
    indicated by number of ones in that row.

    Args:
        y: Array-like with class values to be converted into a matrix
            (integers from 0 to `num_classes - 1`).
        num_classes: Total number of classes. If `None`, this would be inferred
            as `max(y) + 1`.
        dtype: The data type expected by the input. Default: `'float32'`.

    Returns:
        An ordinal regression matrix representation of the input as a NumPy
        array. The class axis is placed last.
    """
    y = np.array(y, dtype="int")
    input_shape = y.shape

    # Shrink the last dimension if the shape is (..., 1).
    if input_shape and input_shape[-1] == 1 and len(input_shape) > 1:
        input_shape = tuple(input_shape[:-1])

    y = y.reshape(-1)
    if not num_classes:
        num_classes = np.max(y) + 1
    n = y.shape[0]
    range_values = np.arange(num_classes - 1)
    range_values = np.tile(np.expand_dims(range_values, 0), [n, 1])
    ordinal = np.zeros((n, num_classes - 1), dtype=dtype)
    ordinal[range_values < np.expand_dims(y, -1)] = 1
    output_shape = input_shape + (num_classes - 1,)
    ordinal = np.reshape(ordinal, output_shape)
    return ordinal

In [ ]:
scores = [1, 3, 5, 2, 6]

to_ordinal(scores)

## Label Conversion

In [ ]:

train_df["label"] = to_ordinal(train_df.score.values).tolist()
valid_df["label"] = to_ordinal(valid_df.score.values).tolist()

feature_mean, feature_std = fit_feature_scaler(train_df)
train_features = transform_features(train_df, feature_mean, feature_std)
valid_features = transform_features(valid_df, feature_mean, feature_std)


# 🎨 | EDA

In [ ]:
# Show distribution of answers using a bar plot
plt.figure(figsize=(8, 4))
df.score.value_counts().plot.bar(color=[cmap(0.0), cmap(0.25), cmap(0.65), cmap(0.9), cmap(1.0)])
plt.xlabel("Score")
plt.ylabel("Count")
plt.title("Score distribution for Train Data")
plt.show()

# Show distribution of essay length using a bar plot
plt.figure(figsize=(8, 4))
df['essay_length'] = df.full_text.map(len)
df.essay_length.plot.hist(logy=False, color=cmap(0.9))
plt.xlabel("Essay Length")
plt.ylabel("Count")
plt.title("Essay Length distribution for Train Data")
plt.show()

# 🍽️ | Preprocessing

**What it does:** The preprocessor takes input strings and transforms them into a dictionary (`token_ids`, `padding_mask`) containing preprocessed tensors. This process starts with tokenization, where input strings are converted into sequences of token IDs.

**Why it's important:** Initially, raw text data is complex and challenging for modeling due to its high dimensionality. By converting text into a compact set of tokens, such as transforming `"The quick brown fox"` into `["the", "qu", "##ick", "br", "##own", "fox"]`, we simplify the data. Many models rely on special tokens and additional tensors to understand input. These tokens help divide input and identify padding, among other tasks. Making all sequences the same length through padding boosts computational efficiency, making subsequent steps smoother.

Explore the following pages to access the available preprocessing and tokenizer layers in **KerasNLP**:
- [Preprocessing](https://keras.io/api/keras_nlp/preprocessing_layers/)
- [Tokenizers](https://keras.io/api/keras_nlp/tokenizers/)

In [ ]:
if CFG.run_original_model and not CFG.use_lgbm:
    preprocessor_cls = getattr(keras_nlp.models, CFG.preprocessor_cls)
    preprocessor = preprocessor_cls.from_preset(
        preset=CFG.preset,
        sequence_length=CFG.sequence_length,
    )
elif not CFG.run_original_model:
    preprocessor = None
    print("Original baseline models disabled by RUN_MODELS; skip original KerasNLP preprocessor.")
else:
    preprocessor = None


Now, let's examine what the output shape of the preprocessing layer looks like. The output shape of the layer can be represented as $(num\_choices, sequence\_length)$.

In [ ]:
if CFG.run_original_model and not CFG.use_lgbm:
    inp = preprocessor(df.full_text.iloc[0])
    for k, v in inp.items():
        print(k, ":", v.shape)
elif not CFG.run_original_model:
    print("Original baseline models disabled by RUN_MODELS; skip preprocessing preview.")
else:
    print("Using LGBM mode; skip KerasNLP preprocessing preview.")


We'll use the `preprocessing_fn` function to transform each text option using the `dataset.map(preprocessing_fn)` method.

In [ ]:

def preprocess_fn(text, features=None, label=None):
    text = preprocessor(text)  # Preprocess text
    if features is not None:
        text = dict(text)
        text["essay_features"] = tf.cast(features, tf.float32)
    return (text, label) if label is not None else text


# 🍚 | DataLoader

The code below sets up a robust data flow pipeline using `tf.data.Dataset` for data processing. Notable aspects of `tf.data` include its ability to simplify pipeline construction and represent components in sequences.

To learn more about `tf.data`, refer to this [documentation](https://www.tensorflow.org/guide/data).

In [ ]:

def build_dataset(texts, features=None, labels=None, batch_size=32,
                  cache=True, drop_remainder=True,
                  shuffle=1024):
    AUTO = tf.data.AUTOTUNE  # AUTOTUNE option
    if features is None and labels is None:
        slices = texts
    elif features is None:
        slices = (texts, labels)
    elif labels is None:
        slices = (texts, features)
    else:
        slices = (texts, features, labels)

    ds = tf.data.Dataset.from_tensor_slices(slices)  # Create dataset from slices
    ds = ds.cache() if cache else ds  # Cache dataset if enabled

    if features is None and labels is None:
        ds = ds.map(lambda text: preprocess_fn(text), num_parallel_calls=AUTO)
    elif features is None:
        ds = ds.map(lambda text, label: preprocess_fn(text, label=label), num_parallel_calls=AUTO)
    elif labels is None:
        ds = ds.map(lambda text, feature: preprocess_fn(text, features=feature), num_parallel_calls=AUTO)
    else:
        ds = ds.map(lambda text, feature, label: preprocess_fn(text, features=feature, label=label), num_parallel_calls=AUTO)

    opt = tf.data.Options()  # Create dataset options
    if shuffle: 
        ds = ds.shuffle(shuffle, seed=CFG.seed)  # Shuffle dataset if enabled
        opt.experimental_deterministic = False
    ds = ds.with_options(opt)  # Set dataset options
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)  # Batch dataset
    ds = ds.prefetch(AUTO)  # Prefetch next batch
    return ds


## Build Train/Valid Dataset

The function below generates the training and validation datasets.

In [ ]:
# Shared train/validation arrays
train_labels = np.array(train_df.label.tolist())
valid_labels = np.array(valid_df.label.tolist())
train_scores = train_df.score.values.astype("float32")
valid_scores = valid_df.score.values.astype("float32")

# LGBM sees the full cleaned text; Transformer sees the head/tail truncated text.
lgbm_train_texts = train_df.full_text_clean.tolist()
lgbm_valid_texts = valid_df.full_text_clean.tolist()
transformer_train_texts = train_df.full_text_transformer.tolist()
transformer_valid_texts = valid_df.full_text_transformer.tolist()

if CFG.run_original_model and not CFG.use_lgbm:
    # Build training dataset
    train_ds = build_dataset(
        transformer_train_texts,
        features=train_features,
        labels=train_labels,
        batch_size=CFG.batch_size,
        shuffle=True,
    )

    # Build validation dataset
    valid_ds = build_dataset(
        transformer_valid_texts,
        features=valid_features,
        labels=valid_labels,
        batch_size=CFG.batch_size,
        shuffle=False,
    )
elif CFG.run_original_model and CFG.use_lgbm:
    train_ds = None
    valid_ds = None
else:
    train_ds = None
    valid_ds = None
    print("Original baseline dataset disabled by RUN_MODELS.")


# ⚓ | LR Schedule

Implementing a learning rate scheduler is crucial for transfer learning. The learning rate initiates at `lr_start` and gradually tapers down to `lr_min` using various techniques, including:
- `step`: Lowering the learning rate in step-wise manner resembling stairs.
- `cos`: Utilizing a cosine curve to gradually reduce the learning rate.
- `exp`: Exponentially decreasing the learning rate.

**Importance:** A well-structured learning rate schedule is essential for efficient model training, ensuring optimal convergence and avoiding issues such as overshooting or stagnation.

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 0.6e-5, 0.3e-5 * batch_size, 0.3e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 2, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

In [ ]:
lr_cb = None if (not CFG.run_original_model or CFG.use_lgbm) else get_lr_callback(CFG.batch_size, plot=True)


# 💾 | Model Checkpointing

The following code will create a callback that will save the best checkpoint of the model during training, which we will use for inference in the submission.

In [ ]:
ckpt_cb = None if (not CFG.run_original_model or CFG.use_lgbm) else keras.callbacks.ModelCheckpoint(
    "best_model.weights.h5",
    monitor="val_weighted_kappa",
    save_best_only=True,
    save_weights_only=True,
    mode="max",
)


# 📏 | Metric

The metric for this competition is quadratic **Weighted Kappa**. This metric is particularly useful for tasks involving ordinal classification (where labels have inherent order). The following code implements this metric from scratch. This metric is implemented taking inspiration from [this TensorFlow implementation](https://www.tensorflow.org/addons/api_docs/python/tfa/losses/WeightedKappaLoss). You can learn more about this metric [here](https://www.sciencedirect.com/science/article/abs/pii/S0167865517301666). 

> This metric implementation is a bit different than the competition metric, which was resolved by @taichiuemura in [here](https://www.kaggle.com/code/taichiuemura/aes-2-0-kerasnlp-starter/#%F0%9F%93%8F-%7C-Metric).

In [ ]:
class WeightedKappa(keras.metrics.Metric):
    def __init__(self, num_classes=6, epsilon=1e-6):
        super().__init__(name="weighted_kappa")
        self.num_classes = num_classes
        self.epsilon = epsilon

        label_vec = keras.ops.arange(num_classes, dtype=keras.backend.floatx())
        self.row_label_vec = keras.ops.reshape(label_vec, [1, num_classes])
        self.col_label_vec = keras.ops.reshape(label_vec, [num_classes, 1])
        col_mat = keras.ops.tile(self.col_label_vec, [1, num_classes])
        row_mat = keras.ops.tile(self.row_label_vec, [num_classes, 1])
        self.weight_mat = (col_mat - row_mat) ** 2

        self.numerator = self.add_weight(name="numerator", initializer="zeros")
        self.denominator = self.add_weight(name="denominator", initializer="zeros")
        self.o_sum = self.add_weight(name="o_sum", initializer="zeros")
        self.e_sum = self.add_weight(name="e_sum", initializer="zeros")

    def update_state(self, y_true, y_pred, **args):
        # revert ordinal regression labels to classification labels
        y_true = keras.ops.one_hot(
            keras.ops.sum(keras.ops.cast(y_true, dtype="int8"), axis=-1) - 1, 6
        )
        y_pred = keras.ops.one_hot(
            keras.ops.sum(keras.ops.cast(y_pred > 0.5, dtype="int8"), axis=-1) - 1, 6
        )
        # weighted kappa calculation
        y_true = keras.ops.cast(y_true, dtype=self.col_label_vec.dtype)
        y_pred = keras.ops.cast(y_pred, dtype=self.weight_mat.dtype)
        batch_size = keras.ops.shape(y_true)[0]

        cat_labels = keras.ops.matmul(y_true, self.col_label_vec)
        cat_label_mat = keras.ops.tile(cat_labels, [1, self.num_classes])
        row_label_mat = keras.ops.tile(self.row_label_vec, [batch_size, 1])

        weight = (cat_label_mat - row_label_mat) ** 2

        self.numerator.assign_add(keras.ops.sum(weight * y_pred))
        label_dist = keras.ops.sum(y_true, axis=0, keepdims=True)
        pred_dist = keras.ops.sum(y_pred, axis=0, keepdims=True)
        w_pred_dist = keras.ops.matmul(
            self.weight_mat, keras.ops.transpose(pred_dist, [1, 0])
        )
        self.denominator.assign_add(
            keras.ops.sum(keras.ops.matmul(label_dist, w_pred_dist))
        )

        self.o_sum.assign_add(keras.ops.sum(y_pred))
        self.e_sum.assign_add(
            keras.ops.sum(
                keras.ops.matmul(keras.ops.transpose(label_dist, [1, 0]), pred_dist)
            )
        )

    def result(self):
        return 1.0 - (
            keras.ops.divide_no_nan(self.numerator, self.denominator)
            * keras.ops.divide_no_nan(self.e_sum, self.o_sum)
        )

    def reset_state(self):
        self.numerator.assign(0)
        self.denominator.assign(0)
        self.o_sum.assign(0)
        self.e_sum.assign(0)


# 🤖 | Modeling



## KerasNLP Classifier

The `KerasNLP` library provides comprehensive, ready-to-use implementations of popular NLP model architectures. It features a variety of pre-trained models including `Bert`, `Roberta`, `DebertaV3`, and more. In this notebook, we'll showcase the usage of `DebertaV3`. However, feel free to explore all available models in the [KerasNLP documentation](https://keras.io/api/keras_nlp/models/). Also, for a deeper understanding of `KerasNLP`, refer to the informative [getting started guide](https://keras.io/guides/keras_nlp/getting_started/).

Our approach involves using `keras_nlp.models.DebertaV3Classifier` to process each text and generate logits. These logits are passed through a `sigmoid` function to produce the final output.

> Note that we are tackling this problem as an **ordinal regression** problem, thus we have used `sigmoid` activation, but we can also consider this problem as a classification problem; then, we would've used `num_classes=6`, `softmax` activation, and `categorical_crossentropy` loss. We could even consider this as a **regression** problem with `num_classes=1`, `no activation` layer, and regression losses like MSE, MAE, etc. You are welcome to experiment with the typical classification and regression methods.

## Screenshot 4 - Model Architecture

**Purpose:** Show the model design. The text model output is fused with essay features, followed by normalization, dense layer, multi-sample dropout, and ordinal regression output.

In [ ]:
from sklearn.metrics import cohen_kappa_score

if not CFG.run_original_model:
    print("Original baseline training disabled by RUN_MODELS. Continue to added-model section.")
    model = None
elif CFG.use_lgbm:
    from scipy.sparse import csr_matrix, hstack
    from scipy.optimize import minimize
    from sklearn.feature_extraction.text import TfidfVectorizer
    import lightgbm as lgb

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=CFG.lgbm_word_max_features,
        strip_accents="unicode",
        sublinear_tf=True,
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=CFG.lgbm_char_max_features,
        sublinear_tf=True,
    )

    def build_lgbm_matrix(texts, features, fit=False):
        if fit:
            word_matrix = word_vectorizer.fit_transform(texts)
            char_matrix = char_vectorizer.fit_transform(texts)
        else:
            word_matrix = word_vectorizer.transform(texts)
            char_matrix = char_vectorizer.transform(texts)
        return hstack([word_matrix, char_matrix, csr_matrix(features)], format="csr")

    def apply_lgbm_thresholds(preds, thresholds):
        thresholds = np.sort(np.asarray(thresholds, dtype="float32"))
        thresholds = np.clip(thresholds, 1.0, 6.0)
        return (np.digitize(preds, thresholds) + 1).clip(1, 6).astype(int)

    def optimize_lgbm_thresholds(y_true, preds):
        initial_thresholds = np.array([1.5, 2.5, 3.5, 4.5, 5.5], dtype="float32")

        def objective(thresholds):
            pred_scores = apply_lgbm_thresholds(preds, thresholds)
            return -cohen_kappa_score(y_true.astype(int), pred_scores, weights="quadratic")

        result = minimize(
            objective,
            initial_thresholds,
            method="Nelder-Mead",
            options={"maxiter": 200, "disp": False},
        )
        if result.success:
            return np.sort(np.clip(result.x, 1.0, 6.0))
        return initial_thresholds

    X_train_lgbm = build_lgbm_matrix(lgbm_train_texts, train_features, fit=True)
    X_valid_lgbm = build_lgbm_matrix(lgbm_valid_texts, valid_features, fit=False)

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=CFG.lgbm_estimators,
        learning_rate=CFG.lgbm_learning_rate,
        num_leaves=31,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=CFG.seed,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(
        X_train_lgbm,
        train_scores,
        eval_set=[(X_valid_lgbm, valid_scores)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(CFG.lgbm_early_stopping_rounds),
            lgb.log_evaluation(50),
        ],
    )

    valid_pred_raw = model.predict(X_valid_lgbm, num_iteration=model.best_iteration_)
    lgbm_thresholds = optimize_lgbm_thresholds(valid_scores, valid_pred_raw) if CFG.lgbm_optimize_thresholds else np.array([1.5, 2.5, 3.5, 4.5, 5.5])
    valid_pred_scores = apply_lgbm_thresholds(valid_pred_raw, lgbm_thresholds)
    valid_qwk = cohen_kappa_score(valid_scores.astype(int), valid_pred_scores, weights="quadratic")
    print(f"LGBM thresholds: {np.round(lgbm_thresholds, 4)}")
    print(f"LGBM valid QWK: {valid_qwk:.5f}")
else:
    # Transformer ordinal model with essay-feature fusion and multi-sample dropout.
    classifier_cls = getattr(keras_nlp.models, CFG.classifier_cls)
    classifier = classifier_cls.from_preset(
        CFG.preset,
        preprocessor=None,
        num_classes=6,
        activation=None,
    )

    text_inputs = classifier.input
    feature_input = keras.Input(shape=(len(FEATURE_COLS),), name="essay_features", dtype="float32")
    text_logits = classifier(text_inputs)

    x = keras.layers.Concatenate(name="text_feature_fusion")([text_logits, feature_input])
    x = keras.layers.BatchNormalization(name="feature_norm")(x)
    x = keras.layers.Dense(CFG.head_units, activation="gelu", name="fusion_dense")(x)

    ordinal_head = keras.layers.Dense(6, name="ordinal_logits", dtype="float32")
    logits = keras.layers.Average(name="multi_sample_dropout")([
        ordinal_head(keras.layers.Dropout(CFG.head_dropout)(x)) for _ in range(CFG.head_samples)
    ])
    outputs = keras.layers.Activation("sigmoid", dtype="float32", name="ordinal_probs")(logits)

    model_inputs = dict(text_inputs)
    model_inputs["essay_features"] = feature_input
    model = keras.Model(model_inputs, outputs)

    # Compile the model with optimizer, loss, and metrics
    model.compile(
        optimizer=keras.optimizers.Adam(5e-6),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[
            WeightedKappa()
        ],
    )
    model.summary()


# 🚂 | Training

Following code will train the model on the training dataset and evaluate on the validation dataset.

In [ ]:
if CFG.run_original_model and not CFG.use_lgbm:
    history = model.fit(
        train_ds,
        epochs=CFG.epochs,
        validation_data=valid_ds,
        callbacks=[lr_cb, ckpt_cb]
    )
elif not CFG.run_original_model:
    history = None
    print("Original baseline training disabled by RUN_MODELS.")
else:
    history = None
    print("LGBM was trained in the previous cell.")


## Result Summary

In [ ]:
if not CFG.run_original_model:
    best_score = np.nan
    print("Original baseline result disabled by RUN_MODELS.")
elif CFG.use_lgbm:
    best_score = valid_qwk
    print(f'\n{"=" * 17} RESULTS {"=" * 17}')
    print(f'>>>> BEST Score : {best_score:.5f}\n>>>> Model      : LGBM')
    print('=' * 50)
else:
    # Find the epoch with the best validation accuracy
    best_epoch = np.argmax(model.history.history['val_weighted_kappa'])
    best_score = model.history.history['val_weighted_kappa'][best_epoch]
    best_loss = model.history.history['val_loss'][best_epoch]

    # Print and display best results
    print(f'\n{"=" * 17} RESULTS {"=" * 17}')
    print(f'>>>> BEST Loss  : {best_loss:.3f}\n>>>> BEST Score : {best_score:.3f}\n>>>> BEST Epoch : {best_epoch}')
    print('=' * 50)


# 🧪 | Testing

In this section, we will visually test how our model performs on some samples from the validation data.

> Note that we are converting the ordinal regression model outputs with `sum`, unlike a typical classification problem where we would use `argmax`.

In [ ]:
if not CFG.run_original_model:
    print("Original baseline prediction preview disabled by RUN_MODELS.")
else:
    if CFG.use_lgbm:
        pred_scores = valid_pred_scores
        true_scores = valid_df.score.values
    else:
        # Load best checkpoint
        model.load_weights("best_model.weights.h5")

        # Make predictions using the trained model on validation data
        valid_preds = model.predict(valid_ds, verbose=0)

        # Format predictions and true answers
        pred_scores = np.sum((valid_preds > 0.5).astype(int), axis=-1).clip(1, 6)
        true_scores = valid_df.score.values

    # Check 5 Predictions
    print("# Predictions\n")
    for i in range(5):
        row = valid_df.iloc[i]
        text = row.full_text
        pred_answer = pred_scores[i]
        true_answer = true_scores[i]
        print(f"Text {i+1}:\n{text[:150]} .... {text[-150:]}\n")
        print(f"True: {true_answer}\n")
        print(f"Predicted: {pred_answer}\n")
        print("-" * 90, "\n")


# 📬 | Submission

In this section, we will infer our model on the test data and then finally prepare the submission file.

## Build Test Dataset

In [ ]:
test_df = pd.read_csv(f"{BASE_PATH}/test.csv")
test_df["full_text_clean"] = test_df["full_text"].fillna("").apply(clean_text)
test_df["full_text_transformer"] = test_df["full_text_clean"].apply(truncate_head_tail)
test_df = add_essay_features(test_df)

test_features = transform_features(test_df, feature_mean, feature_std)

if CFG.run_original_model and CFG.use_lgbm:
    lgbm_test_texts = test_df.full_text_clean.tolist()
    X_test_lgbm = build_lgbm_matrix(lgbm_test_texts, test_features, fit=False)
    test_ds = None
elif CFG.run_original_model and not CFG.use_lgbm:
    transformer_test_texts = test_df.full_text_transformer.tolist()
    test_ds = build_dataset(
        transformer_test_texts,
        features=test_features,
        labels=None,
        batch_size=min(CFG.batch_size, len(test_df)),
        shuffle=False,
        drop_remainder=False,
    )
else:
    test_ds = None
    print("Original baseline test dataset disabled by RUN_MODELS; added models will build their own test loaders.")


## Extra Transformer Model Comparison

Kaggle-only comparison block for the three added inputs now shown in your notebook: Longformer-base, ELECTRA, and ALBERT. Turn `HF_CFG.enabled = True` to run them.

In [ ]:
# ===== Huang Po-Shuo Section 1: Extra Model Experiment Settings =====
# Purpose: compare three added models on Kaggle and tune speed/accuracy parameters.
class HF_CFG:
    # Set True when you want to run the three added model inputs on Kaggle.
    enabled = bool(globals().get("HF_ENABLED_MODELS", []))
    # Keep False for Kaggle submissions. Add models through the right-side Add Input panel.
    allow_internet_download = False
    skip_missing_models = True

    # Three added inputs currently in your Kaggle sidebar.
    # Models used for my comparison experiment.
    compare_models = globals().get("HF_ENABLED_MODELS", ["longformer", "electra", "albert"])
    submission_model = globals().get("HF_SUBMISSION_MODEL", None) or (compare_models[0] if compare_models else None)

    # Same idea as the original model: ordinal target + essay-feature fusion + multi-sample dropout.
    epochs = 2
    grad_accum_steps = 4
    num_workers = 2
    use_amp = True
    head_units = 64
    head_dropout = 0.25
    head_samples = 5

    # Use None for full data. Put 512/256 here only for a quick smoke test.
    max_train_samples = None
    max_valid_samples = None

    # Manual paths are optional. Auto-detect reads the Kaggle Inputs under /kaggle/input.
    # Fill only if auto-detect chooses the wrong folder.
    local_model_paths = {
        "longformer": None,
    }

    model_settings = {
        "longformer": {
            "engine": "hf_torch",
            "hf_name": "allenai/longformer-base-4096",
            "input_keywords": ["longformer", "base"],
            # Parameter tuning: faster Kaggle setting, about 4x fewer train steps than 1024/batch1/2 epochs.
            "max_length": 768,
            "batch_size": 2,
            "epochs": 1,
            "grad_accum_steps": 2,
            "lr": 1.2e-5,
            "use_full_text": True,
        },
        "electra": {
            "engine": "keras_nlp",
            "preset": "electra_large_discriminator_uncased_en",
            "tokenizer_cls": "ElectraTokenizer",
            "backbone_cls": "ElectraBackbone",
            "max_length": 512,
            "batch_size": 4,
            "lr": 5e-6,
            "use_full_text": False,
        },
        "albert": {
            "engine": "keras_nlp",
            "preset": "albert_large_en_uncased",
            "preprocessor_cls": "AlbertTextClassifierPreprocessor",
            "backbone_cls": "AlbertBackbone",
            # Memory-safe Kaggle setting for ALBERT large.
            "max_length": 256,
            "batch_size": 1,
            "lr": 3e-6,
            "use_full_text": False,
        },
    }


In [ ]:
# ===== Huang Po-Shuo Section 2: Train and Evaluate Added Models =====
# Purpose: run Longformer / ELECTRA / ALBERT and record validation QWK for comparison.
import os
import pandas as pd

if HF_CFG.enabled:
    import gc
    import math
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
    from sklearn.metrics import cohen_kappa_score

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("HF/PyTorch device:", device)

    def list_kaggle_hf_candidates():
        rows = []
        input_root = "/kaggle/input"
        if not os.path.exists(input_root):
            return pd.DataFrame(columns=["path", "has_config", "has_tokenizer", "has_weights"])
        for root, dirs, files in os.walk(input_root):
            lower_files = {f.lower() for f in files}
            has_config = "config.json" in lower_files
            has_tokenizer = any(name in lower_files for name in ["tokenizer.json", "vocab.txt", "spiece.model", "merges.txt"])
            has_weights = any(f.endswith((".bin", ".safetensors", ".h5")) for f in lower_files)
            if has_config or has_tokenizer or has_weights:
                rows.append({
                    "path": root,
                    "has_config": has_config,
                    "has_tokenizer": has_tokenizer,
                    "has_weights": has_weights,
                })
        return pd.DataFrame(rows)

    def is_valid_hf_local_dir(path):
        if path is None:
            return False
        path = str(path)
        if not os.path.exists(path):
            return False
        files = {f.lower() for f in os.listdir(path)}
        return "config.json" in files and any(
            name in files for name in ["tokenizer.json", "vocab.txt", "spiece.model", "merges.txt"]
        ) and any(f.endswith((".bin", ".safetensors")) for f in files)

    def find_hf_model_path(model_key, settings):
        local_model_paths = getattr(HF_CFG, "local_model_paths", {})
        manual_path = local_model_paths.get(model_key)
        if is_valid_hf_local_dir(manual_path):
            print("Using manual Kaggle model input:", manual_path)
            return manual_path

        keywords = [k.lower().replace("_", "-") for k in settings["input_keywords"]]
        candidates_df = list_kaggle_hf_candidates()
        if len(candidates_df):
            scored = []
            for path in candidates_df["path"].tolist():
                lower_path = path.lower().replace("_", "-")
                score = sum(k in lower_path for k in keywords)
                if score > 0 and is_valid_hf_local_dir(path):
                    scored.append((score, len(path), path))
            if scored:
                scored = sorted(scored, key=lambda x: (-x[0], x[1]))
                print("Using auto-detected Kaggle HF input:", scored[0][2])
                return scored[0][2]

        message = (
            f"Missing local Kaggle Input for {model_key}: {settings['hf_name']}. "
            "Add it with Add Input, or set HF_CFG.local_model_paths manually."
        )
        if getattr(HF_CFG, "allow_internet_download", False):
            print(message)
            print("Internet download is enabled; trying HuggingFace name:", settings["hf_name"])
            return settings["hf_name"]
        raise FileNotFoundError(message)

    class HFEssayDataset(Dataset):
        def __init__(self, texts, features, labels, tokenizer, max_length, model_name_or_path):
            self.texts = list(texts)
            self.features = np.asarray(features, dtype="float32")
            self.labels = None if labels is None else np.asarray(labels, dtype="float32")
            self.tokenizer = tokenizer
            self.max_length = max_length
            self.is_longformer = "longformer" in str(model_name_or_path).lower()

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            encoded = self.tokenizer(
                self.texts[idx],
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
            )
            item = {k: v.squeeze(0) for k, v in encoded.items()}
            if self.is_longformer:
                global_attention_mask = torch.zeros_like(item["attention_mask"])
                global_attention_mask[0] = 1
                item["global_attention_mask"] = global_attention_mask
            item["essay_features"] = torch.tensor(self.features[idx], dtype=torch.float32)
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
            return item

    class HFOrdinalFeatureModel(nn.Module):
        def __init__(self, model_name_or_path, feature_dim, head_units=64, head_dropout=0.25, head_samples=5):
            super().__init__()
            config = AutoConfig.from_pretrained(model_name_or_path)
            self.backbone = AutoModel.from_pretrained(model_name_or_path, config=config)
            hidden_size = config.hidden_size
            self.norm = nn.LayerNorm(hidden_size + feature_dim)
            self.fusion = nn.Linear(hidden_size + feature_dim, head_units)
            self.activation = nn.GELU()
            self.dropouts = nn.ModuleList([nn.Dropout(head_dropout) for _ in range(head_samples)])
            self.ordinal_head = nn.Linear(head_units, 6)

        def forward(self, essay_features=None, labels=None, **batch):
            outputs = self.backbone(**batch)
            pooled = outputs.pooler_output if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]
            x = torch.cat([pooled, essay_features], dim=1)
            x = self.activation(self.fusion(self.norm(x)))
            logits = torch.stack([self.ordinal_head(dropout(x)) for dropout in self.dropouts]).mean(dim=0)
            loss = None if labels is None else nn.BCEWithLogitsLoss()(logits, labels)
            return loss, logits

    def hf_scores_from_logits(logits):
        probs = torch.sigmoid(torch.tensor(logits)).numpy()
        return np.sum((probs > 0.5).astype(int), axis=-1).clip(1, 6).astype(int)

    def hf_evaluate(model, loader):
        model.eval()
        losses, logits_list, labels_list = [], [], []
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop("labels").to(device)
                features = batch.pop("essay_features").to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                loss, logits = model(**batch, essay_features=features, labels=labels)
                losses.append(loss.item())
                logits_list.append(logits.detach().cpu().numpy())
                labels_list.append(labels.detach().cpu().numpy())
        logits = np.concatenate(logits_list)
        labels = np.concatenate(labels_list)
        pred_scores = hf_scores_from_logits(logits)
        true_scores = labels.sum(axis=1).clip(1, 6).astype(int)
        qwk = cohen_kappa_score(true_scores, pred_scores, weights="quadratic")
        return float(np.mean(losses)), float(qwk)

    def hf_predict(model, loader):
        model.eval()
        logits_list = []
        with torch.no_grad():
            for batch in loader:
                # Validation batches include labels; test batches do not. Remove labels for prediction.
                batch.pop("labels", None)
                features = batch.pop("essay_features").to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                _, logits = model(**batch, essay_features=features, labels=None)
                logits_list.append(logits.detach().cpu().numpy())
        logits = np.concatenate(logits_list)
        return hf_scores_from_logits(logits), logits

    # Longformer uses HuggingFace/PyTorch because the Kaggle input is a PyTorch model.
    def run_hf_model(model_key):
        settings = HF_CFG.model_settings[model_key]
        model_path = find_hf_model_path(model_key, settings)
        print(f"\n===== HF/PyTorch model: {model_key} | {model_path} =====")

        train_texts = train_df.full_text_clean.tolist() if settings["use_full_text"] else train_df.full_text_transformer.tolist()
        valid_texts = valid_df.full_text_clean.tolist() if settings["use_full_text"] else valid_df.full_text_transformer.tolist()
        test_texts = test_df.full_text_clean.tolist() if settings["use_full_text"] else test_df.full_text_transformer.tolist()

        local_train_features = train_features
        local_valid_features = valid_features
        local_train_labels = train_labels
        local_valid_labels = valid_labels
        if HF_CFG.max_train_samples is not None:
            idx = np.random.RandomState(CFG.seed).choice(len(train_texts), min(HF_CFG.max_train_samples, len(train_texts)), replace=False)
            train_texts = [train_texts[i] for i in idx]
            local_train_features = local_train_features[idx]
            local_train_labels = local_train_labels[idx]
        if HF_CFG.max_valid_samples is not None:
            idx = np.random.RandomState(CFG.seed + 1).choice(len(valid_texts), min(HF_CFG.max_valid_samples, len(valid_texts)), replace=False)
            valid_texts = [valid_texts[i] for i in idx]
            local_valid_features = local_valid_features[idx]
            local_valid_labels = local_valid_labels[idx]

        tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
        train_ds = HFEssayDataset(train_texts, local_train_features, local_train_labels, tokenizer, settings["max_length"], model_path)
        valid_ds = HFEssayDataset(valid_texts, local_valid_features, local_valid_labels, tokenizer, settings["max_length"], model_path)
        test_ds = HFEssayDataset(test_texts, test_features, None, tokenizer, settings["max_length"], model_path)

        train_loader = DataLoader(train_ds, batch_size=settings["batch_size"], shuffle=True, num_workers=HF_CFG.num_workers, pin_memory=True)
        valid_loader = DataLoader(valid_ds, batch_size=settings["batch_size"], shuffle=False, num_workers=HF_CFG.num_workers, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=settings["batch_size"], shuffle=False, num_workers=HF_CFG.num_workers, pin_memory=True)

        model = HFOrdinalFeatureModel(
            model_path,
            feature_dim=len(FEATURE_COLS),
            head_units=HF_CFG.head_units,
            head_dropout=HF_CFG.head_dropout,
            head_samples=HF_CFG.head_samples,
        ).to(device)

        model_epochs = settings.get("epochs", HF_CFG.epochs)
        grad_accum_steps = settings.get("grad_accum_steps", HF_CFG.grad_accum_steps)
        optimizer = torch.optim.AdamW(model.parameters(), lr=settings["lr"])
        total_steps = math.ceil(len(train_loader) / grad_accum_steps) * model_epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=(HF_CFG.use_amp and device.type == "cuda"))

        best_qwk = -1
        best_path = f"best_{model_key}.pt"
        for epoch in range(model_epochs):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            running_loss = 0.0
            for step, batch in enumerate(train_loader, start=1):
                labels = batch.pop("labels").to(device)
                features = batch.pop("essay_features").to(device)
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=(HF_CFG.use_amp and device.type == "cuda")):
                    loss, _ = model(**batch, essay_features=features, labels=labels)
                    loss = loss / grad_accum_steps
                scaler.scale(loss).backward()
                running_loss += loss.item() * grad_accum_steps
                if step % grad_accum_steps == 0 or step == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                if step % 100 == 0:
                    print(f"{model_key} epoch {epoch + 1} step {step}/{len(train_loader)} loss {running_loss / step:.4f}")

            valid_loss, valid_qwk = hf_evaluate(model, valid_loader)
            print(f"{model_key} epoch {epoch + 1}: valid_loss={valid_loss:.4f}, valid_qwk={valid_qwk:.5f}")
            if valid_qwk > best_qwk:
                best_qwk = valid_qwk
                torch.save(model.state_dict(), best_path)

        model.load_state_dict(torch.load(best_path, map_location=device))
        valid_pred_scores, valid_raw_logits = hf_predict(model, valid_loader)
        test_pred, test_raw_logits = hf_predict(model, test_loader)
        del model, tokenizer, train_loader, valid_loader, test_loader
        gc.collect()
        torch.cuda.empty_cache()
        return {"model": model_key, "valid_qwk": best_qwk, "test_pred": test_pred, "valid_raw": valid_raw_logits, "test_raw": test_raw_logits}

    def build_keras_extra_dataset(texts, features=None, labels=None, preprocessor=None, batch_size=4,
                                  cache=False, drop_remainder=False, shuffle=False):
        AUTO = tf.data.AUTOTUNE
        if features is None and labels is None:
            slices = texts
        elif features is None:
            slices = (texts, labels)
        elif labels is None:
            slices = (texts, features)
        else:
            slices = (texts, features, labels)
        ds = tf.data.Dataset.from_tensor_slices(slices)
        ds = ds.cache() if cache else ds

        def map_text(text, feature=None, label=None):
            inputs = dict(preprocessor(text))
            if feature is not None:
                inputs["essay_features"] = tf.cast(feature, tf.float32)
            return (inputs, label) if label is not None else inputs

        if features is None and labels is None:
            ds = ds.map(lambda text: map_text(text), num_parallel_calls=AUTO)
        elif features is None:
            ds = ds.map(lambda text, label: map_text(text, label=label), num_parallel_calls=AUTO)
        elif labels is None:
            ds = ds.map(lambda text, feature: map_text(text, feature=feature), num_parallel_calls=AUTO)
        else:
            ds = ds.map(lambda text, feature, label: map_text(text, feature=feature, label=label), num_parallel_calls=AUTO)
        if shuffle:
            ds = ds.shuffle(1024, seed=CFG.seed)
        return ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(AUTO)

    def keras_probs_to_scores(probs):
        return np.sum((probs > 0.5).astype(int), axis=-1).clip(1, 6).astype(int)

    def keras_probs_to_raw_scores(probs):
        # Continuous ordinal score used for validation-threshold tuning.
        return np.sum(np.asarray(probs, dtype="float32"), axis=-1).clip(1, 6)

    def apply_score_thresholds(raw_scores, thresholds):
        thresholds = np.sort(np.asarray(thresholds, dtype="float32"))
        thresholds = np.clip(thresholds, 1.0, 6.0)
        return (np.digitize(raw_scores, thresholds) + 1).clip(1, 6).astype(int)

    def optimize_score_thresholds(y_true, raw_scores):
        from scipy.optimize import minimize
        initial_thresholds = np.array([1.5, 2.5, 3.5, 4.5, 5.5], dtype="float32")

        def objective(thresholds):
            pred_scores = apply_score_thresholds(raw_scores, thresholds)
            return -cohen_kappa_score(y_true.astype(int), pred_scores, weights="quadratic")

        result = minimize(
            objective,
            initial_thresholds,
            method="Nelder-Mead",
            options={"maxiter": 200, "disp": False},
        )
        if result.success:
            return np.sort(np.clip(result.x, 1.0, 6.0))
        return initial_thresholds

    def get_pooled_output(backbone_outputs):
        if isinstance(backbone_outputs, (tuple, list)):
            return backbone_outputs[-1]
        if isinstance(backbone_outputs, dict):
            for key in ["pooled_output", "pooler_output", "sequence_output"]:
                if key in backbone_outputs:
                    out = backbone_outputs[key]
                    return out[:, 0] if len(out.shape) == 3 else out
        return backbone_outputs[:, 0] if len(backbone_outputs.shape) == 3 else backbone_outputs

    def make_electra_preprocessor(settings):
        tokenizer_cls = getattr(keras_nlp.models, settings["tokenizer_cls"])
        tokenizer = tokenizer_cls.from_preset(settings["preset"])
        cls_id = tokenizer.token_to_id("[CLS]")
        sep_id = tokenizer.token_to_id("[SEP]")
        pad_id = tokenizer.token_to_id("[PAD]")
        packer = keras_nlp.layers.MultiSegmentPacker(
            sequence_length=settings["max_length"],
            start_value=cls_id,
            end_value=sep_id,
            pad_value=pad_id,
        )

        def preprocessor(text):
            token_ids, segment_ids = packer(tokenizer(text))
            padding_mask = tf.cast(token_ids != pad_id, tf.int32)
            return {
                "token_ids": token_ids,
                "segment_ids": segment_ids,
                "padding_mask": padding_mask,
            }
        return preprocessor

    def make_keras_preprocessor(settings):
        if settings["preset"].startswith("electra"):
            return make_electra_preprocessor(settings)
        preprocessor_cls = getattr(keras_nlp.models, settings["preprocessor_cls"])
        return preprocessor_cls.from_preset(
            settings["preset"],
            sequence_length=settings["max_length"],
        )

    def build_keras_extra_model(settings):
        preproc = make_keras_preprocessor(settings)
        backbone_cls = getattr(keras_nlp.models, settings["backbone_cls"])
        backbone = backbone_cls.from_preset(settings["preset"])

        text_inputs = {
            "token_ids": keras.Input(shape=(settings["max_length"],), dtype="int32", name="token_ids"),
            "segment_ids": keras.Input(shape=(settings["max_length"],), dtype="int32", name="segment_ids"),
            "padding_mask": keras.Input(shape=(settings["max_length"],), dtype="int32", name="padding_mask"),
        }
        feature_input = keras.Input(shape=(len(FEATURE_COLS),), name="essay_features", dtype="float32")
        pooled = get_pooled_output(backbone(text_inputs))
        x = keras.layers.Concatenate(name="text_feature_fusion")([pooled, feature_input])
        x = keras.layers.BatchNormalization(name="feature_norm")(x)
        x = keras.layers.Dense(HF_CFG.head_units, activation="gelu", name="fusion_dense")(x)
        ordinal_head = keras.layers.Dense(6, name="ordinal_logits", dtype="float32")
        logits = keras.layers.Average(name="multi_sample_dropout")([
            ordinal_head(keras.layers.Dropout(HF_CFG.head_dropout)(x)) for _ in range(HF_CFG.head_samples)
        ])
        outputs = keras.layers.Activation("sigmoid", dtype="float32", name="ordinal_probs")(logits)
        model_inputs = dict(text_inputs)
        model_inputs["essay_features"] = feature_input
        model = keras.Model(model_inputs, outputs)
        model.compile(
            optimizer=keras.optimizers.Adam(settings["lr"]),
            loss=keras.losses.BinaryCrossentropy(),
            metrics=[WeightedKappa()],
        )
        return preproc, model

    # ELECTRA and ALBERT use Kaggle Keras model inputs.
    def run_kerasnlp_model(model_key):
        settings = HF_CFG.model_settings[model_key]
        print(f"\n===== KerasNLP model: {model_key} | {settings['preset']} =====")
        local_train_texts = train_df.full_text_clean.tolist() if settings["use_full_text"] else train_df.full_text_transformer.tolist()
        local_valid_texts = valid_df.full_text_clean.tolist() if settings["use_full_text"] else valid_df.full_text_transformer.tolist()
        local_test_texts = test_df.full_text_clean.tolist() if settings["use_full_text"] else test_df.full_text_transformer.tolist()
        local_train_features = train_features
        local_valid_features = valid_features
        local_train_labels = train_labels
        local_valid_labels = valid_labels
        if HF_CFG.max_train_samples is not None:
            idx = np.random.RandomState(CFG.seed).choice(len(local_train_texts), min(HF_CFG.max_train_samples, len(local_train_texts)), replace=False)
            local_train_texts = [local_train_texts[i] for i in idx]
            local_train_features = local_train_features[idx]
            local_train_labels = local_train_labels[idx]
        if HF_CFG.max_valid_samples is not None:
            idx = np.random.RandomState(CFG.seed + 1).choice(len(local_valid_texts), min(HF_CFG.max_valid_samples, len(local_valid_texts)), replace=False)
            local_valid_texts = [local_valid_texts[i] for i in idx]
            local_valid_features = local_valid_features[idx]
            local_valid_labels = local_valid_labels[idx]

        preproc, extra_model = build_keras_extra_model(settings)
        extra_train_ds = build_keras_extra_dataset(local_train_texts, local_train_features, local_train_labels, preproc, settings["batch_size"], cache=False, drop_remainder=True, shuffle=True)
        extra_valid_ds = build_keras_extra_dataset(local_valid_texts, local_valid_features, local_valid_labels, preproc, settings["batch_size"], cache=False, drop_remainder=False, shuffle=False)
        extra_test_ds = build_keras_extra_dataset(local_test_texts, test_features, None, preproc, min(settings["batch_size"], len(test_df)), cache=False, drop_remainder=False, shuffle=False)

        ckpt_path = f"best_{model_key}.weights.h5"
        callbacks = [
            get_lr_callback(settings["batch_size"], epochs=HF_CFG.epochs, plot=False),
            keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_weighted_kappa", save_best_only=True, save_weights_only=True, mode="max"),
        ]
        history = extra_model.fit(extra_train_ds, epochs=HF_CFG.epochs, validation_data=extra_valid_ds, callbacks=callbacks)
        extra_model.load_weights(ckpt_path)
        valid_probs = extra_model.predict(extra_valid_ds, verbose=0)
        test_probs = extra_model.predict(extra_test_ds, verbose=0)
        true_scores = local_valid_labels.sum(axis=1).clip(1, 6).astype(int)

        valid_raw_scores = keras_probs_to_raw_scores(valid_probs)
        test_raw_scores = keras_probs_to_raw_scores(test_probs)
        score_thresholds = optimize_score_thresholds(true_scores, valid_raw_scores)

        valid_scores_pred = apply_score_thresholds(valid_raw_scores, score_thresholds)
        valid_qwk = cohen_kappa_score(true_scores, valid_scores_pred, weights="quadratic")
        test_pred = apply_score_thresholds(test_raw_scores, score_thresholds)

        print(f"{model_key} tuned thresholds:", np.round(score_thresholds, 4))
        print(f"{model_key} validation prediction distribution:", dict(pd.Series(valid_scores_pred).value_counts().sort_index()))
        print(f"{model_key} test prediction distribution:", dict(pd.Series(test_pred).value_counts().sort_index()))
        del extra_model, preproc, extra_train_ds, extra_valid_ds, extra_test_ds
        gc.collect()
        keras.backend.clear_session()
        return {"model": model_key, "valid_qwk": float(valid_qwk), "test_pred": test_pred, "valid_raw": valid_probs, "test_raw": test_probs}

    def run_extra_model(model_key):
        settings = HF_CFG.model_settings[model_key]
        if settings.get("engine") == "keras_nlp":
            return run_kerasnlp_model(model_key)
        return run_hf_model(model_key)


In [ ]:
# ===== Huang Po-Shuo Section 3: Display Model Comparison Results =====
# Purpose: collect each model's validation QWK and show a comparison table.
import os
import pandas as pd

HF_RESULTS = []
HF_TEST_PREDS_BY_MODEL = {}
HF_VALID_RAW_BY_MODEL = {}
HF_TEST_RAW_BY_MODEL = {}
HF_ERRORS = {}

if HF_CFG.enabled:
    if "list_kaggle_hf_candidates" in globals():
        print("Available Kaggle HF-like model candidates for Longformer:")
        display(list_kaggle_hf_candidates().head(30))
    for model_key in HF_CFG.compare_models:
        try:
            result = run_extra_model(model_key)
            HF_RESULTS.append({"model": result["model"], "valid_qwk": result["valid_qwk"]})
            HF_TEST_PREDS_BY_MODEL[result["model"]] = result["test_pred"]
            HF_VALID_RAW_BY_MODEL[result["model"]] = result["valid_raw"]
            HF_TEST_RAW_BY_MODEL[result["model"]] = result["test_raw"]
        except Exception as exc:
            HF_ERRORS[model_key] = repr(exc)
            if getattr(HF_CFG, "skip_missing_models", True):
                print(f"[SKIP] {model_key}: {exc}")
            else:
                raise

    hf_comparison_df = pd.DataFrame(HF_RESULTS).sort_values("valid_qwk", ascending=False) if HF_RESULTS else pd.DataFrame(columns=["model", "valid_qwk"])
    display(hf_comparison_df)
    if HF_ERRORS:
        print("Extra model errors:")
        for model_key, message in HF_ERRORS.items():
            print(model_key, "->", message)
else:
    print("Extra model comparison disabled. Set HF_CFG.enabled = True to run Longformer/ELECTRA/ALBERT.")


## LGBM + Transformer Ensemble

Optional cell to blend LGBM raw predictions with HuggingFace Transformer ordinal predictions and re-optimize QWK thresholds. Run after the HF comparison cell.

In [ ]:
# ===== Huang Po-Shuo Optional Section: Strongest Optimized Model Ensemble =====
# Goal: strongest Kaggle submission from available reliable models.
# Recommended switch board: LGBM=True, Longformer=True, ELECTRA=True, ALBERT=False.
# This cell searches single models, pair blends, and 3-model blends, then refines the best weights.

class ENS_CFG:
    enabled = True
    max_models = 3
    coarse_weight_step = 0.05
    refine_top_k = 8
    refine_radius = 0.08
    refine_weight_step = 0.01
    min_single_valid_qwk = 0.72
    exclude_models = ["albert"]


def optimize_qwk_thresholds(y_true, raw_preds):
    initial_thresholds = np.array([1.5, 2.5, 3.5, 4.5, 5.5], dtype="float32")

    def objective(thresholds):
        pred_scores = apply_lgbm_thresholds(raw_preds, thresholds)
        return -cohen_kappa_score(y_true.astype(int), pred_scores, weights="quadratic")

    result = minimize(
        objective,
        initial_thresholds,
        method="Nelder-Mead",
        options={"maxiter": 220, "disp": False},
    )
    if result.success:
        return np.sort(np.clip(result.x, 1.0, 6.0))
    return initial_thresholds


def ordinal_logits_to_raw_scores(outputs):
    # Longformer stores logits; KerasNLP ELECTRA/ALBERT store probabilities.
    outputs = np.asarray(outputs, dtype="float32")
    if outputs.ndim == 1:
        return outputs.clip(1, 6)
    if outputs.min() >= 0.0 and outputs.max() <= 1.0:
        probs = outputs
    else:
        probs = 1.0 / (1.0 + np.exp(-outputs))
    return probs.sum(axis=1).clip(1, 6)


def integer_weight_vectors(n_models, total_units):
    if n_models == 1:
        yield (total_units,)
        return
    if n_models == 2:
        for first in range(total_units + 1):
            yield (first, total_units - first)
        return
    for first in range(total_units + 1):
        for rest in integer_weight_vectors(n_models - 1, total_units - first):
            yield (first,) + rest


def normalized_local_weight_vectors(center, radius, step):
    center = np.asarray(center, dtype="float32")
    n_models = len(center)
    units = np.arange(-radius, radius + 1e-9, step, dtype="float32")
    if n_models == 1:
        yield np.array([1.0], dtype="float32")
        return
    if n_models == 2:
        lo = max(0.0, center[0] - radius)
        hi = min(1.0, center[0] + radius)
        for first in np.arange(lo, hi + 1e-9, step):
            yield np.array([first, 1.0 - first], dtype="float32")
        return
    for a in units:
        for b in units:
            candidate = center.copy()
            candidate[0] += a
            candidate[1] += b
            candidate[2] = 1.0 - candidate[0] - candidate[1]
            if np.all(candidate >= -1e-6) and np.all(candidate <= 1.0 + 1e-6):
                yield np.clip(candidate, 0.0, 1.0).astype("float32")


def collect_ensemble_candidates():
    candidates = {}

    if all(name in globals() for name in ["valid_pred_raw", "X_test_lgbm", "model", "lgbm_thresholds"]):
        candidates["lgbm"] = {
            "valid_raw": np.asarray(valid_pred_raw, dtype="float32"),
            "test_raw": np.asarray(model.predict(X_test_lgbm, num_iteration=model.best_iteration_), dtype="float32"),
        }

    if "HF_VALID_RAW_BY_MODEL" in globals() and "HF_TEST_RAW_BY_MODEL" in globals():
        for model_name, valid_raw in HF_VALID_RAW_BY_MODEL.items():
            if model_name in HF_TEST_RAW_BY_MODEL:
                candidates[model_name] = {
                    "valid_raw": ordinal_logits_to_raw_scores(valid_raw),
                    "test_raw": ordinal_logits_to_raw_scores(HF_TEST_RAW_BY_MODEL[model_name]),
                }

    return candidates


def evaluate_blend(combo, weights, candidates, y_valid):
    blend_valid_raw = np.zeros_like(candidates[combo[0]]["valid_raw"], dtype="float32")
    for model_name, weight in zip(combo, weights):
        blend_valid_raw += float(weight) * candidates[model_name]["valid_raw"]
    thresholds = optimize_qwk_thresholds(y_valid, blend_valid_raw)
    pred_scores = apply_lgbm_thresholds(blend_valid_raw, thresholds)
    qwk = cohen_kappa_score(y_valid, pred_scores, weights="quadratic")
    return float(qwk), thresholds


ENSEMBLE_RESULT = None
ensemble_test_preds = None
ensemble_search_df = pd.DataFrame()

if ENS_CFG.enabled:
    if "valid_scores" not in globals():
        raise RuntimeError("valid_scores is missing. Run the train/validation split cells before ensemble.")

    y_valid = valid_scores.astype(int)
    all_candidates = collect_ensemble_candidates()
    if not all_candidates:
        raise RuntimeError("No ensemble candidates found. Enable LGBM and at least one added model first.")

    usable_candidates = {}
    single_rows = []
    for name, values in all_candidates.items():
        thresholds = optimize_qwk_thresholds(y_valid, values["valid_raw"])
        pred_scores = apply_lgbm_thresholds(values["valid_raw"], thresholds)
        qwk = cohen_kappa_score(y_valid, pred_scores, weights="quadratic")
        single_rows.append({"model": name, "single_valid_qwk": float(qwk)})
        if name not in ENS_CFG.exclude_models and qwk >= ENS_CFG.min_single_valid_qwk:
            usable_candidates[name] = values

    single_model_df = pd.DataFrame(single_rows).sort_values("single_valid_qwk", ascending=False).reset_index(drop=True)
    print("Single-model validation check for ensemble candidates:")
    display(single_model_df)

    if not usable_candidates:
        raise RuntimeError("All candidates were filtered out. Lower ENS_CFG.min_single_valid_qwk or check model outputs.")

    rows = []
    best = None
    model_names = list(usable_candidates.keys())
    total_units = int(round(1.0 / ENS_CFG.coarse_weight_step))

    from itertools import combinations
    for n_models in range(1, min(ENS_CFG.max_models, len(model_names)) + 1):
        for combo in combinations(model_names, n_models):
            for units in integer_weight_vectors(n_models, total_units):
                weights = np.array(units, dtype="float32") / total_units
                qwk, thresholds = evaluate_blend(combo, weights, usable_candidates, y_valid)
                weight_map = {model_name: float(weight) for model_name, weight in zip(combo, weights)}
                row = {
                    "stage": "coarse",
                    "models": "+".join(combo),
                    "valid_qwk": qwk,
                    **{f"w_{model_name}": weight_map.get(model_name, 0.0) for model_name in model_names},
                }
                rows.append(row)
                if best is None or qwk > best["valid_qwk"]:
                    best = {"models": combo, "weights": weights, "valid_qwk": qwk, "thresholds": thresholds}

    coarse_df = pd.DataFrame(rows).sort_values("valid_qwk", ascending=False).reset_index(drop=True)
    print("Top coarse ensemble candidates:")
    display(coarse_df.head(20))

    # Fine search around the strongest coarse candidates.
    refine_rows = []
    for _, coarse_row in coarse_df.head(ENS_CFG.refine_top_k).iterrows():
        combo = tuple(coarse_row["models"].split("+"))
        center = np.array([coarse_row.get(f"w_{name}", 0.0) for name in combo], dtype="float32")
        for weights in normalized_local_weight_vectors(center, ENS_CFG.refine_radius, ENS_CFG.refine_weight_step):
            qwk, thresholds = evaluate_blend(combo, weights, usable_candidates, y_valid)
            weight_map = {model_name: float(weight) for model_name, weight in zip(combo, weights)}
            row = {
                "stage": "fine",
                "models": "+".join(combo),
                "valid_qwk": qwk,
                **{f"w_{model_name}": weight_map.get(model_name, 0.0) for model_name in model_names},
            }
            refine_rows.append(row)
            if qwk > best["valid_qwk"]:
                best = {"models": combo, "weights": weights, "valid_qwk": qwk, "thresholds": thresholds}

    ensemble_search_df = pd.concat([coarse_df, pd.DataFrame(refine_rows)], ignore_index=True)
    ensemble_search_df = ensemble_search_df.sort_values("valid_qwk", ascending=False).reset_index(drop=True)
    print("Top final ensemble candidates:")
    display(ensemble_search_df.head(20))

    blend_test_raw = np.zeros_like(usable_candidates[best["models"][0]]["test_raw"], dtype="float32")
    for model_name, weight in zip(best["models"], best["weights"]):
        blend_test_raw += float(weight) * usable_candidates[model_name]["test_raw"]
    ensemble_test_preds = apply_lgbm_thresholds(blend_test_raw, best["thresholds"])

    ENSEMBLE_RESULT = best
    print("Best strongest ensemble:")
    print({
        "models": list(best["models"]),
        "weights": {m: round(float(w), 4) for m, w in zip(best["models"], best["weights"])},
        "valid_qwk": round(best["valid_qwk"], 5),
        "thresholds": np.round(best["thresholds"], 4).tolist(),
    })
else:
    print("Ensemble disabled. Set ENS_CFG.enabled = True after running LGBM and added models.")


## Inference on Test Data

## Submission Output

**Screenshot focus:** After selecting the best available model, this block generates prediction scores for the Kaggle test set. In the ELECTRA experiment, the notebook prints `Using HF submission model: electra`.

In [ ]:
# ===== Huang Po-Shuo Section 4: Select Model for Submission =====
# Priority: ensemble first, then selected added model, then LGBM/original transformer.
if "ENSEMBLE_RESULT" in globals() and ENSEMBLE_RESULT is not None and "ensemble_test_preds" in globals():
    test_preds = ensemble_test_preds
    print("Using LGBM + Transformer ensemble submission")
elif HF_CFG.enabled and HF_TEST_PREDS_BY_MODEL and HF_CFG.submission_model in HF_TEST_PREDS_BY_MODEL:
    test_preds = HF_TEST_PREDS_BY_MODEL[HF_CFG.submission_model]
    print(f"Using HF submission model: {HF_CFG.submission_model}")
elif HF_CFG.enabled and HF_TEST_PREDS_BY_MODEL:
    fallback_model = sorted(HF_TEST_PREDS_BY_MODEL.keys())[0]
    test_preds = HF_TEST_PREDS_BY_MODEL[fallback_model]
    print(f"HF submission_model was not available; using fallback HF model: {fallback_model}")
elif CFG.run_original_model and CFG.use_lgbm:
    test_preds = model.predict(X_test_lgbm, num_iteration=model.best_iteration_)
    test_preds = apply_lgbm_thresholds(test_preds, lgbm_thresholds)
elif CFG.run_original_model:
    # Do inference
    test_preds = model.predict(test_ds, verbose=1)

    # Convert probabilities to class labels
    test_preds = np.sum((test_preds > 0.5).astype(int), axis=-1).clip(1, 6)
else:
    raise RuntimeError("No model selected. Enable one RUN_MODELS switch before creating submission.")


## Create Submission File

In [ ]:
# ===== Huang Po-Shuo Section 5: Create Kaggle Submission File =====
# Create a DataFrame to store the submission
sub_df = test_df[["essay_id"]].copy()

# Add the formatted predictions to the submission DataFrame
sub_df["score"] = test_preds

# Save Submission
sub_df.to_csv('submission.csv',index=False)

# Display the first 2 rows of the submission DataFrame
sub_df.head()

## Final Model Comparison

Run this after training/inference to summarize all candidate models in one table. Fill `public_score` and `private_score` after each Kaggle submission if needed.

## Final Model Comparison Table

**Screenshot focus:** This table summarizes all model experiments for the report, including LGBM, original KerasNLP models, and the three added models. Leaderboard scores can be filled after each Kaggle submission.

In [ ]:
# ===== Huang Po-Shuo Section 6: Final Report Comparison Table =====
# This table is safe to run even when only one model is enabled.

MODEL_COMPARISON_TEMPLATE = [
    {
        "group": "Traditional ML",
        "model": "LGBM + TF-IDF + essay features",
        "run_key": "lgbm",
        "preset_or_hf_name": "LightGBMRegressor",
        "sequence_length": "full text",
        "notes": "Current strongest baseline",
    },
    {
        "group": "KerasNLP original",
        "model": "BERT base",
        "run_key": "bert",
        "preset_or_hf_name": "bert_base_en_uncased",
        "sequence_length": getattr(CFG, "sequence_length", np.nan),
        "notes": "Original Kaggle input model",
    },
    {
        "group": "KerasNLP original",
        "model": "RoBERTa base",
        "run_key": "roberta",
        "preset_or_hf_name": "roberta_base_en",
        "sequence_length": getattr(CFG, "sequence_length", np.nan),
        "notes": "Original Kaggle input model",
    },
    {
        "group": "KerasNLP original",
        "model": "DeBERTaV3 small / extra small",
        "run_key": "deberta_v3_small",
        "preset_or_hf_name": "deberta_v3_small_en or deberta_v3_extra_small_en",
        "sequence_length": getattr(CFG, "sequence_length", np.nan),
        "notes": "Original Kaggle input model",
    },
    {
        "group": "KerasNLP original",
        "model": "DeBERTaV3 base",
        "run_key": "deberta_v3_base",
        "preset_or_hf_name": "deberta_v3_base_en",
        "sequence_length": getattr(CFG, "sequence_length", np.nan),
        "notes": "Original Kaggle input model",
    },
    {
        "group": "HF/PyTorch added",
        "model": "Longformer base",
        "run_key": "longformer",
        "preset_or_hf_name": "allenai/longformer-base-4096",
        "sequence_length": HF_CFG.model_settings.get("longformer", {}).get("max_length", 768) if "HF_CFG" in globals() else 768,
        "notes": "Added long-text Transformer",
    },
    {
        "group": "KerasNLP added",
        "model": "ALBERT large",
        "run_key": "albert",
        "preset_or_hf_name": "albert_large_en_uncased",
        "sequence_length": HF_CFG.model_settings.get("albert", {}).get("max_length", 512) if "HF_CFG" in globals() else 512,
        "notes": "Added official Kaggle Keras model",
    },
    {
        "group": "KerasNLP added",
        "model": "ELECTRA large discriminator",
        "run_key": "electra",
        "preset_or_hf_name": "electra_large_discriminator_uncased_en",
        "sequence_length": HF_CFG.model_settings.get("electra", {}).get("max_length", 512) if "HF_CFG" in globals() else 512,
        "notes": "Added official Kaggle Keras model",
    },
]

LEADERBOARD_SCORES = {
    "lgbm": {"public_score": 0.79506, "private_score": 0.80488},
    "roberta": {"public_score": 0.76231, "private_score": 0.78513},
    "electra": {"public_score": 0.75016, "private_score": 0.78383},
}

validation_scores = {}
if "valid_qwk" in globals() and isinstance(globals().get("valid_qwk"), (int, float, np.floating)):
    validation_scores["lgbm"] = float(valid_qwk)

if "best_score" in globals() and pd.notna(globals().get("best_score", np.nan)) and getattr(CFG, "run_original_model", False) and not getattr(CFG, "use_lgbm", False):
    active_key = getattr(CFG, "active_original_model", None)
    if active_key:
        validation_scores[active_key] = float(best_score)

for row in globals().get("HF_RESULTS", []):
    if isinstance(row, dict) and "model" in row and "valid_qwk" in row:
        validation_scores[row["model"]] = float(row["valid_qwk"])

rows = []
for row in MODEL_COMPARISON_TEMPLATE:
    key = row["run_key"]
    out = dict(row)
    out["valid_qwk"] = validation_scores.get(key, np.nan)
    out["public_score"] = LEADERBOARD_SCORES.get(key, {}).get("public_score", np.nan)
    out["private_score"] = LEADERBOARD_SCORES.get(key, {}).get("private_score", np.nan)
    out["status"] = "ran_this_session" if key in validation_scores else "pending / fill after version run"
    rows.append(out)

all_model_comparison_df = pd.DataFrame(rows)[[
    "group",
    "model",
    "preset_or_hf_name",
    "sequence_length",
    "valid_qwk",
    "public_score",
    "private_score",
    "status",
    "notes",
]]

ranked_model_comparison_df = all_model_comparison_df.sort_values(
    by=["private_score", "public_score", "valid_qwk"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display(ranked_model_comparison_df)


# ✍️ | Reference
* [LLM Science Exam: KerasCore + KerasNLP [TPU]](https://www.kaggle.com/code/awsaf49/llm-science-exam-kerascore-kerasnlp-tpu)
* [Keras NLP](https://keras.io/api/keras_nlp/)
* [Triple Stratified KFold with TFRecords](https://www.kaggle.com/code/cdeotte/triple-stratified-kfold-with-tfrecords) by @cdeotte